In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import joblib
import datetime
import sklearn

from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.cluster import KMeans
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score


print("\n==============================")
print("LOADING DATASET")
print("==============================\n")

df = pd.read_csv("Salary.csv")
df = df.drop_duplicates().reset_index(drop=True)

print("Dataset shape:", df.shape)


# --------------------------------------------------
# FEATURE ENGINEERING
# --------------------------------------------------

def add_career_score(X):

    X = X.copy()

    X["Career Score"] = (
        X["Years of Experience"] +
        (X["Education Level"] * 2)
    )

    return X[
        [
            "Years of Experience",
            "Education Level",
            "Career Score"
        ]
    ]


features = [
    "Years of Experience",
    "Education Level"
]

X = df[features]


# --------------------------------------------------
# FINAL CLUSTER COUNT
# --------------------------------------------------

K_FINAL = 3


# --------------------------------------------------
# PIPELINE
# --------------------------------------------------

pipeline = Pipeline([
    ("career_score", FunctionTransformer(add_career_score)),
    ("scaler", StandardScaler()),
    ("kmeans", KMeans(
        n_clusters=K_FINAL,
        random_state=42,
        n_init=20
    ))
])


print("\nTraining clustering pipeline...\n")

labels = pipeline.fit_predict(X)

df["Cluster"] = labels


# --------------------------------------------------
# TRANSFORMED FEATURES
# --------------------------------------------------

X_engineered = pipeline.named_steps["career_score"].transform(X)

scaler = pipeline.named_steps["scaler"]

X_scaled = scaler.transform(X_engineered)


# --------------------------------------------------
# CLUSTER QUALITY METRICS
# --------------------------------------------------

sil_score = silhouette_score(X_scaled, labels)

db_score = davies_bouldin_score(X_scaled, labels)

print("\nCluster Quality Metrics")

print("Silhouette Score:", sil_score)

print("Davies-Bouldin Score:", db_score)


# --------------------------------------------------
# CLUSTER STATISTICS
# --------------------------------------------------

print("\nCluster sizes:\n")

cluster_sizes = df["Cluster"].value_counts().sort_index()

print(cluster_sizes)

print("\nCluster statistics:\n")

cluster_stats = df.groupby("Cluster")[
    ["Years of Experience", "Education Level", "Salary"]
].mean()

print(cluster_stats)


# --------------------------------------------------
# DETERMINE CAREER STAGE MAPPING
# --------------------------------------------------

cluster_experience = cluster_stats["Years of Experience"].sort_values()

career_stage_labels = [
    "Entry Stage",
    "Growth Stage",
    "Leadership Stage"
]

cluster_to_stage = {}

for i, cluster_id in enumerate(cluster_experience.index):
    cluster_to_stage[int(cluster_id)] = career_stage_labels[i]

print("\nCluster → Career Stage Mapping")

print(cluster_to_stage)


# --------------------------------------------------
# CLUSTER CENTROIDS
# --------------------------------------------------

centroids = pipeline.named_steps["kmeans"].cluster_centers_

centroids_df = pd.DataFrame(
    centroids,
    columns=["Years of Experience", "Education Level", "Career Score"]
)

print("\nCluster Centroids")

print(centroids_df)


# --------------------------------------------------
# CREATE OUTPUT DIRECTORIES
# --------------------------------------------------

os.makedirs("cluster_plots", exist_ok=True)

os.makedirs("model", exist_ok=True)


# --------------------------------------------------
# PCA VISUALIZATION
# --------------------------------------------------

pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(7,5))

plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    c=df["Cluster"],
    cmap="viridis",
    alpha=0.6
)

plt.title("Career Clusters (PCA Projection)")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")

plt.savefig("cluster_plots/pca_clusters.png")

plt.close()


# --------------------------------------------------
# CLUSTER SIZE PLOT
# --------------------------------------------------

plt.figure(figsize=(6,4))

cluster_sizes.plot(kind="bar")

plt.title("Cluster Size Distribution")
plt.xlabel("Cluster")
plt.ylabel("Count")

plt.savefig("cluster_plots/cluster_sizes.png")

plt.close()


# --------------------------------------------------
# EXPERIENCE VS SALARY
# --------------------------------------------------

plt.figure(figsize=(7,5))

plt.scatter(
    df["Years of Experience"],
    df["Salary"],
    c=df["Cluster"],
    cmap="viridis",
    alpha=0.5
)

plt.title("Experience vs Salary by Cluster")
plt.xlabel("Years of Experience")
plt.ylabel("Salary")

plt.savefig("cluster_plots/experience_salary_clusters.png")

plt.close()


# --------------------------------------------------
# CAREER SCORE DISTRIBUTION
# --------------------------------------------------

df["Career Score"] = (
    df["Years of Experience"] +
    (df["Education Level"] * 2)
)

plt.figure(figsize=(7,5))

for c in sorted(df["Cluster"].unique()):

    subset = df[df["Cluster"] == c]

    plt.hist(
        subset["Career Score"],
        bins=20,
        alpha=0.5,
        label=f"Cluster {c}"
    )

plt.title("Career Score Distribution per Cluster")
plt.xlabel("Career Score")
plt.ylabel("Frequency")

plt.legend()

plt.savefig("cluster_plots/career_score_distribution.png")

plt.close()


# --------------------------------------------------
# EXPERIENCE DISTRIBUTION
# --------------------------------------------------

plt.figure(figsize=(7,5))

for c in sorted(df["Cluster"].unique()):

    subset = df[df["Cluster"] == c]

    plt.hist(
        subset["Years of Experience"],
        bins=20,
        alpha=0.5,
        label=f"Cluster {c}"
    )

plt.title("Experience Distribution per Cluster")
plt.xlabel("Years of Experience")
plt.ylabel("Frequency")

plt.legend()

plt.savefig("cluster_plots/experience_distribution_clusters.png")

plt.close()


# --------------------------------------------------
# METADATA
# --------------------------------------------------

metadata = {

    "model_type": "KMeans Career Clustering Pipeline",

    "training_dataset": "Salary.csv",

    "dataset_shape": df.shape,

    "features_used": features,

    "engineered_feature": "Career Score = Experience + (Education Level * 2)",

    "cluster_count": K_FINAL,

    "cluster_sizes": cluster_sizes.to_dict(),

    "cluster_statistics": cluster_stats.to_dict(),

    "cluster_stage_mapping": cluster_to_stage,

    "cluster_centroids": centroids_df.to_dict(),

    "silhouette_score": sil_score,

    "davies_bouldin_score": db_score,

    "scaler_mean": scaler.mean_.tolist(),

    "scaler_scale": scaler.scale_.tolist(),

    "scikit_learn_version": sklearn.__version__,

    "training_timestamp": str(datetime.datetime.now())
}


# --------------------------------------------------
# SAVE MODEL PACKAGE
# --------------------------------------------------

package = {
    "model": pipeline,
    "metadata": metadata
}

joblib.dump(
    package,
    "model/career_cluster_pipeline.pkl"
)


print("\nModel package saved:")
print("model/career_cluster_pipeline.pkl")

print("\nSaved Metadata\n")

for key, value in metadata.items():
    print(key, ":", value)


print("\nPlots saved in cluster_plots/")

print("\n==============================")
print("CLUSTER TRAINING COMPLETE")
print("==============================\n")
